# Opdracht 1: het huidige behandelproces in Python
Klik op een codeblok en druk op **Shift + Enter**. Werk van boven naar beneden en selecteer dezelfde Python-kernel als in het startnotebook voor data-analyse.

Dit notebook is een zelfstandige leerversie van `procesmodel.py`: je hoeft het Python-bestand niet eerst uit te voeren. Er zijn geen extra Python-pakketten nodig voor de modelcode.

**Doel:** de processtappen uit de case herkenbaar weergeven, met dezelfde codes als in het visuele model van je teamgenoot. De invoer bestaat uit zelfgekozen scenario's. De code stelt geen diagnose en voorspelt niets.

Bron: pagina 1 en opdracht 1 op pagina 2 van [de case](../../informatie/Heart%20Failure%20Prediction%20Case%20incl%20ethics.pdf). [Uitgebreide toelichting en aannames](../../informatie/opdracht1/README.md).

## Feiten en modelkeuzes
De case noemt binnenkomst via huisarts of SEH, eerste zorg door verpleegkundigen, directe IC bij ernstige gevallen met evident hartinfarct, onderzoek door specialisten en diagnose door een artsenteam. Daarna worden IC-verblijf, reguliere opname, direct vertrek, operatie en overlijden genoemd.

De case geeft geen medische criteria voor de vervolgroutes. We voeren die keuzes daarom zelf in. De volgorde van operatie en overlijden is onbekend: die tonen we als losse gebeurtenissen. De hoofdroute tot diagnose volgt de tekst; eventuele overplaatsingen en heronderzoeken zijn niet uitgewerkt. We kiezen per voorbeeld één vervolgroute; dit is een vereenvoudiging, geen ziekenhuisprotocol.

## 1. Geef elk procesonderdeel een code
Een **dictionary** koppelt een sleutel, zoals `'S01'`, aan een omschrijving. Zo gebruiken de code en het visuele model dezelfde labels. `S` duidt een activiteit of verblijfsstap aan, `D` een beslispunt en `E` een gebeurtenis.

**E01 en E02 zijn alternatieve startgebeurtenissen:** binnenkomst via huisarts of SEH. De parameter `instroom` beschrijft via welke route dat gebeurt: huisarts of SEH. De eerste zorg door verpleegkundigen is vervolgens activiteit S01. E03 is overlijden, waarvan het tijdstip niet is beschreven.

De naam `STAPPEN` blijft behouden als verzamelnaam voor alle procesonderdelen, dus ook gebeurtenissen en beslispunten. Deze codes zijn onze modelkeuze; dit is geen volledige formele BPMN-uitwerking.

Voer de cel uit. Er verschijnt nog geen uitvoer: Python bewaart de dictionary onder de naam `STAPPEN`.

D01 en D02 blijven als labels beschikbaar voor vergelijking met het visuele model. Ze staan niet in de routelijst: de uitvoer gebruikt ze als onderbouwing onder de relevante stap. Een onderbouwing beschrijft een ingevoerde keuze; waar medische criteria ontbreken, benoemen we dat.


In [1]:
# Een dictionary koppelt elke unieke procescode aan een leesbare omschrijving.
STAPPEN = {
    # E01 en E02 zijn alternatieve startgebeurtenissen: de gekozen instroom bepaalt welke wordt gebruikt.
    'E01': 'Binnenkomst via huisarts',
    'E02': 'Binnenkomst via spoedeisende hulp',
    # S-codes beschrijven activiteiten of verblijf; ze kunnen in de routelijst voorkomen.
    'S01': 'Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie',
    # D-codes zijn labels voor onderbouwing en worden niet als aparte routestap afgedrukt.
    'D01': 'Ernstig geval met evident hartinfarct?',
    'S02': 'Direct naar intensive care',
    'S03': 'Specialisten meten patiëntwaarden (meestal enkele uren)',
    'S04': 'Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies',
    'D02': 'Vervolgroute volgens gekozen scenario (criteria niet beschreven)',
    'S05': 'Verblijf op intensive care',
    'S06': 'Reguliere ziekenhuisopname',
    'S07': 'Ziekenhuis direct verlaten',
    # Operatie en overlijden worden apart bewaard: hun tijdstip is niet vastgelegd in de case.
    'S08': 'Operatie, bijvoorbeeld plaatsing van stents',
    'E03': 'Overlijden (tijdstip in het proces onbekend)',
}


## 2. Vraag één omschrijving op
Met vierkante haken zoek je de waarde bij een sleutel op. `print()` toont die waarde. Verwacht hieronder de omschrijving van de eerste zorg door verpleegkundigen (S01).

In [2]:
# Zoek met de sleutel S01 de omschrijving op in STAPPEN en toon die met print().
print(STAPPEN['S01'])  # Output: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie


Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie


## 3. Bouw een route met een functie
Met `def` definieer je een functie: herbruikbare code die pas werkt wanneer je haar aanroept.

De invoerparameters zijn:
- `instroom`: `'huisarts'` of `'seh'`; de aankomstroute bij startgebeurtenis E01 of E02.
- `ernstig`: `True` of `False`, voor een ernstig geval met evident hartinfarct. Dit is een ingevoerde beoordeling.
- `vervolg`: `'ic'`, `'regulier'`, `'vertrek'` of `'overlijden'`.
- `operatie=False`: standaard geen operatie in het scenario; met `True` registreer je die wel.

Lees de functie in drie delen:
1. De eerste `if`-regels controleren of de invoer toegestane waarden heeft. `raise ValueError` meldt ongeldige invoer. Dit controleert geen medische geschiktheid.
2. `route` begint met startgebeurtenis E01 of E02 en activiteit S01. Besliscodes staan niet in deze lijst. `append()` voegt één stap toe; `extend()` voegt meerdere stappen toe. Alleen bij `ernstig=True` wordt S02 toegevoegd.
3. `gebeurtenissen` bewaart operatie en overlijden apart. `return` geeft beide lijsten terug.

`None` betekent hier dat er geen bekende verblijf-/vertrekstap wordt toegevoegd. Bij overlijden kennen we die route niet uit de case. Ook kennen we het moment van overlijden niet; de voorbeeldroute is geen bewering dat iedereen eerst de diagnose bereikt.

In [3]:
# Definieer de functie. operatie=False is de standaardwaarde als je dit argument weglaat.
def modelleer_proces(instroom, ernstig, vervolg, operatie=False):
    """Geef een route en losse gebeurtenissen terug uit handmatige invoer.

    `ernstig` betekent hier exact: ernstig geval met evident hartinfarct.
    `vervolg` is ic, regulier, vertrek of overlijden.
    Een onbekende vervolgroute voorkomt dat we bij overlijden verblijf verzinnen.
    """
    # Controleer of de instroom een van de twee toegestane tekstwaarden is.
    if instroom not in ('huisarts', 'seh'):
        # Stop deze functie met een begrijpelijke foutmelding bij ongeldige invoer.
        raise ValueError('Instroom moet huisarts of seh zijn.')
    # bool is het Python-type voor True en False; or betekent dat een van beide fouten voldoende is.
    if type(ernstig) is not bool or type(operatie) is not bool:
        # Tekst zoals "True" is geen boolean en wordt hier dus afgewezen.
        raise ValueError('Ernstig en operatie moeten True of False zijn.')
    # Vertaal de ingevoerde vervolgkeuze naar een procescode met een dictionary.
    vervolgstappen = {
        'ic': 'S05', 'regulier': 'S06', 'vertrek': 'S07',
        # None betekent: geen bekende verblijf-/vertrekstap. Overlijden wordt later apart geregistreerd.
        'overlijden': None,
    }
    # Bij een dictionary controleert in of de waarde tussen de sleutels staat.
    if vervolg not in vervolgstappen:
        # Een onbekende sleutel mag niet verderop worden gebruikt om een stap op te zoeken.
        raise ValueError('Onbekende vervolgroute.')

    # == vergelijkt twee waarden. = in de volgende regel kent juist een waarde toe.
    if instroom == 'huisarts':
        # Start een lijst met aankomst via de huisarts en de eerste zorg.
        route = ['E01', 'S01']
    # De andere toegestane instroom gebruikt een andere startgebeurtenis.
    if instroom == 'seh':
        # Start een lijst met aankomst via de SEH en dezelfde eerste zorg.
        route = ['E02', 'S01']
    # Voer de ingesprongen regel alleen uit als ernstig True is; er wordt geen diagnose berekend.
    if ernstig:
        # append() voegt een element achteraan toe: directe IC volgens de ingevoerde ernst.
        route.append('S02')
    # extend() voegt meerdere elementen toe: onderzoek en diagnose volgen in beide routes.
    route.extend(['S03', 'S04'])
    # Zoek de procescode op; voeg alleen een vervolg toe als die code bekend is.
    if vervolgstappen[vervolg] is not None:
        # De vierkante haken zoeken de stapcode op bij de ingevoerde vervolgkeuze.
        route.append(vervolgstappen[vervolg])

    # Deze gebeurtenissen krijgen geen verzonnen plek in de tijdlijn.
    # Maak een lege lijst voor losse onderdelen met onbekende timing, ook de operatieactiviteit.
    gebeurtenissen = []
    # Alleen een expliciet gekozen operatie wordt geregistreerd.
    if operatie:
        # Bewaar de operatie buiten de geordende hoofdroute.
        gebeurtenissen.append('S08')
    # Dit is een scenario-uitkomst, geen medische behandelbeslissing.
    if vervolg == 'overlijden':
        # Registreer overlijden zonder te bepalen wanneer het plaatsvindt.
        gebeurtenissen.append('E03')
    # Geef beide lijsten terug aan de code die de functie heeft aangeroepen.
    return route, gebeurtenissen


## 4. Bekijk wat de functie teruggeeft
We roepen de functie aan voor een zelfgekozen voorbeeld. De twee resultaten bewaren we in `route` en `gebeurtenissen`. Verwacht een lijst zonder S02 en een lege gebeurtenissenlijst `[]`.

De routelijst bevat gebeurtenissen en activiteiten, geen D01 of D02. De volgende functie voegt bij het afdrukken de onderbouwing toe.


In [4]:
# Roep de functie aan en verdeel de twee teruggegeven lijsten over twee variabelen.
route, gebeurtenissen = modelleer_proces('huisarts', False, 'vertrek')
# Toon de procescodes in de volgorde waarin de functie ze heeft toegevoegd.
print('Route:', route)
# Een lege lijst [] betekent dat dit scenario geen losse onderdelen heeft.
print('Losse gebeurtenissen:', gebeurtenissen)


Route: ['E01', 'S01', 'S03', 'S04', 'S07']
Losse gebeurtenissen: []


## 5. Toon stappen met hun onderbouwing
De functie roept `modelleer_proces()` aan. Daarna bouwt zij een dictionary `onderbouwing`: de sleutel is de stapcode, de waarde is de uitleg waarom die stap in deze route voorkomt.

- Bij `ernstig=True` krijgt S02 de onderbouwing van D01: de case beschrijft directe IC voor ernstige gevallen met evident hartinfarct.
- Bij `ernstig=False` wordt S02 overgeslagen. In deze versie wordt daarbij geen aparte D01-toelichting afgedrukt.
- S05, S06 of S07 krijgt de ingevoerde vervolgkeuze als toelichting van D02. Dit is geen medische rechtvaardiging: die criteria ontbreken in de case.
- De `else`-tak bevat een reserve-uitleg bij S04. Deze wordt met de huidige toegestane invoer niet bereikt. Bij overlijden wordt een eigen toelichting onder E03 afgedrukt.

`for stap in route:` loopt door de route. `if stap in onderbouwing:` controleert of bij die stap uitleg hoort. Die uitleg wordt ingesprongen onder de stap afgedrukt; een beslissing verschijnt dus niet meer als eigen processtap.

Een **f-string** vult waarden in tekst in. In `{vervolg!r}` zorgt `!r` ervoor dat de tekstwaarde met aanhalingstekens verschijnt, zoals `'ic'`.

De timing van operatie en overlijden is niet vastgelegd. Bij overlijden staat dit ook in de omschrijving. De positie onder de afgedrukte route legt geen tijdsvolgorde vast. De operatie is een activiteit met onbekende timing; overlijden is een gebeurtenis.


In [5]:
# Deze functie verzorgt de presentatie; de route wordt door modelleer_proces gemaakt.
def toon_scenario(naam, instroom, ernstig, vervolg, operatie=False):
    # Geef de scenario-invoer door en ontvang de route en de losse onderdelen.
    route, gebeurtenissen = modelleer_proces(instroom, ernstig, vervolg, operatie)

    # Koppel de onderbouwing aan de stap waarvoor de keuze relevant is.
    # Maak een lege dictionary: straks is de sleutel een stapcode en de waarde de toelichting.
    onderbouwing = {}
    # In deze versie krijgt alleen de ja-route van D01 een expliciete onderbouwing.
    if ernstig:
        # Koppel de uitleg aan de directe IC-stap. Tekstregels binnen de haakjes vormen samen een tekst.
        onderbouwing['S02'] = (
            'D01: ernstig=True ingevoerd: ernstig geval met evident hartinfarct. '
            'Volgens de case volgt dan directe IC.'
        )


    # Koppel de scenario-uitkomst aan een code om de onderbouwing bij die code te bewaren.
    vervolgkeuzes = {'ic': 'S05', 'regulier': 'S06', 'vertrek': 'S07', 'overlijden': 'E03'}
    # Controleer of er voor deze keuze een code is opgenomen.
    if vervolg in vervolgkeuzes:
        # Bewaar de gevonden procescode tijdelijk onder de naam stap.
        stap = vervolgkeuzes[vervolg]
        # Sla de uitleg bij die code op. !r toont de tekstwaarde met aanhalingstekens.
        onderbouwing[stap] = (
            f'D02: vervolg={vervolg!r} is vooraf gekozen in dit scenario. '
            'De case geeft hiervoor geen medische besliscriteria.'
        )
    # Reservepad: met de huidige invoercontrole zijn alle geldige keuzes hierboven al afgedekt.
    else:
        # Deze toelichting zou na de diagnose staan als een geldige keuze geen koppeling had.
        onderbouwing['S04'] = (
            'D02: na deze diagnose is in het scenario geen verblijf-/vertrekroute '
            'ingevuld.'
        )

    # Een f-string vult waarden tussen {} in; \n begint de uitvoer op een nieuwe regel.
    print(f'\n{naam} | instroom: {instroom} | ernstig: {ernstig}')
    # Loop in volgorde door de route; stap bevat telkens een andere procescode.
    for stap in route:
        # Zoek de omschrijving bij de huidige code op en druk beide af.
        print(f'  {stap}: {STAPPEN[stap]}')
        # Print alleen een toelichting als deze stap een sleutel in onderbouwing heeft.
        if stap in onderbouwing:
            # Extra spaties laten zien dat de uitleg bij de bovenstaande stap hoort.
            print(f'    Onderbouwing - {onderbouwing[stap]}')
    # Toon de losse onderdelen. Hun afdrukvolgorde legt geen werkelijk tijdstip vast.
    for stap in gebeurtenissen:
        # Zoek de omschrijving bij de huidige code op en druk beide af.
        print(f'  {stap}: {STAPPEN[stap]}')

        # De operatie heeft een eigen toelichting op basis van de handmatige invoer.
        if stap == 'S08':
            # De code geeft geen medische reden voor een operatie: die criteria ontbreken in de case.
            print('    Onderbouwing - operatie=True is ingevoerd; de case geeft geen operatiecriteria.')

        # Overlijden krijgt hier een aparte uitleg; de eerder opgeslagen D02-tekst wordt hier niet gebruikt.
        if stap == 'E03':
            # Benoem de ingevoerde uitkomst en de ontbrekende informatie over het tijdstip.
            print('    Onderbouwing - vervolg=overlijden is ingevoerd; '
                  'de case geeft geen criteria voor overlijdenstijdstip.')



## 6. Voorbeeld A: via de huisarts, daarna vertrek
Dit is een gekozen procesroute, geen advies om een patiënt te ontslaan. Omdat `ernstig=False`, verschijnt S02 niet. Het gekozen vervolg `'vertrek'` geeft S07.

In [6]:
# Argumenten op volgorde: naam, instroom, ernstig, vervolg; operatie blijft standaard False.
# Voorbeeld A: via huisarts, geen directe IC, daarna vertrek (vooraf gekozen scenario).
toon_scenario('Voorbeeld A', 'huisarts', False, 'vertrek')


Voorbeeld A | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  S07: Ziekenhuis direct verlaten
    Onderbouwing - D02: vervolg='vertrek' is vooraf gekozen in dit scenario. De case geeft hiervoor geen medische besliscriteria.


## 7. Voorbeeld B: directe IC en een operatie
Bij `ernstig=True` verschijnt S02 vóór het onderzoek. Het vervolg `'ic'` voegt S05 toe: IC-verblijf na de diagnose in dit scenario. S02 en S05 hebben dus een verschillende betekenis.

`operatie=True` registreert S08 apart. De case specificeert de timing van die operatie niet.

In [7]:
# Voorbeeld B: via SEH, directe IC bij ernstig=True en vervolgens IC-verblijf.
# operatie=True is een benoemd argument: registreer daarnaast een operatie met onbekende timing.
toon_scenario('Voorbeeld B', 'seh', True, 'ic', operatie=True)


Voorbeeld B | instroom: seh | ernstig: True
  E02: Binnenkomst via spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  S02: Direct naar intensive care
    Onderbouwing - D01: ernstig=True ingevoerd: ernstig geval met evident hartinfarct. Volgens de case volgt dan directe IC.
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  S05: Verblijf op intensive care
    Onderbouwing - D02: vervolg='ic' is vooraf gekozen in dit scenario. De case geeft hiervoor geen medische besliscriteria.
  S08: Operatie, bijvoorbeeld plaatsing van stents
    Onderbouwing - operatie=True is ingevoerd; de case geeft geen operatiecriteria.


## 8. Voorbeeld C: reguliere opname
Verwacht S06 als vervolg. Vergelijk deze uitvoer met voorbeeld A: dezelfde beginroute, maar een andere ingevoerde vervolgkeuze.

In [8]:
# Voorbeeld C: via huisarts, geen directe IC en reguliere opname als gekozen vervolg.
toon_scenario('Voorbeeld C', 'huisarts', False, 'regulier')


Voorbeeld C | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  S06: Reguliere ziekenhuisopname
    Onderbouwing - D02: vervolg='regulier' is vooraf gekozen in dit scenario. De case geeft hiervoor geen medische besliscriteria.


## 9. Voorbeeld D: overlijden genoemd, route onvolledig bekend
`overlijden` is onze technische markering. Overlijden is geen behandelbeslissing. We kennen de plaats in het proces niet en voegen geen verzonnen verblijfroute toe.

Dit eenvoudige voorbeeld toont wel de gebruikelijke hoofdroute; het legt niet vast welke stappen werkelijk aan een overlijden voorafgaan. Het model is op dit punt bewust onvolledig.

In [9]:
# Voorbeeld D: via SEH en directe IC; overlijden staat apart met onbekend tijdstip.
# Dit voorbeeld zegt niet dat elke overleden patient eerst alle stappen heeft doorlopen.
toon_scenario('Voorbeeld D', 'seh', True, 'overlijden')


Voorbeeld D | instroom: seh | ernstig: True
  E02: Binnenkomst via spoedeisende hulp
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  S02: Direct naar intensive care
    Onderbouwing - D01: ernstig=True ingevoerd: ernstig geval met evident hartinfarct. Volgens de case volgt dan directe IC.
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  E03: Overlijden (tijdstip in het proces onbekend)
    Onderbouwing - vervolg=overlijden is ingevoerd; de case geeft geen criteria voor overlijdenstijdstip.


## 10. Probeer zelf een wijziging
Verander hieronder alleen `'regulier'` in `'ic'`. Voorspel eerst de uitvoer en voer daarna de cel uit.

Welke stap verandert? Verschijnt S02 ook? Leg uit welke invoer S02 bepaalt en welke invoer S05 bepaalt.

In [10]:
# Jouw oefening: ernstig=False slaat directe IC (S02) over.
# vervolg="ic" voegt wel IC-verblijf (S05) toe. Deze twee invoerkeuzes staan los van elkaar.
toon_scenario('Mijn oefening', 'huisarts', False, 'ic')


Mijn oefening | instroom: huisarts | ernstig: False
  E01: Binnenkomst via huisarts
  S01: Urgente opname: verpleegkundigen verzorgen eerste zorg en informatie
  S03: Specialisten meten patiëntwaarden (meestal enkele uren)
  S04: Artsenteam beoordeelt gegevens en stelt diagnose op basis van ervaring en studies
  S05: Verblijf op intensive care
    Onderbouwing - D02: vervolg='ic' is vooraf gekozen in dit scenario. De case geeft hiervoor geen medische besliscriteria.


## Mijn bevindingen
Dubbelklik op deze tekst, vul je antwoorden in en druk op **Shift + Enter**.

- Ik verwachtte dat S02 niet verschijnt. 
- In de uitvoer veranderde ...
- S02 wordt bepaald door ...
- S05 wordt bepaald door ...
- Een aanname die ik met mijn teamgenoot wil bespreken is ...

## Wat je nu kunt toelichten
Je gebruikt een dictionary voor stapomschrijvingen, lijsten voor routes en een functie met `if`-regels voor vooraf gekozen vertakkingen. De code berekent geen medische beslissingen. Dezelfde stapcodes kunnen in het visuele model staan.

Dit notebook en `procesmodel.py` zijn afzonderlijk uitvoerbare versies. Wijzigingen worden niet automatisch tussen beide gesynchroniseerd. Gebruik het notebook om te leren en stem inhoudelijke wijzigingen later af met het script.